In [35]:
import chromadb
import json
from dotenv import load_dotenv

from langchain.text_splitter import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer
from tqdm import tqdm

load_dotenv()
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [19]:
section_page_data = json.load(open("../Data/kaggle_section_records.json", "r", encoding="utf-8"))
section_page_data[0]

{'id': 1,
 'context': 'Preface Welcome to Psychology 2e, an OpenStax resource. This textbook was written to increase student access to high-quality learning materials, maintaining highest standards of academic rigor at little to no cost. About OpenStax OpenStax is a nonprofit based at Rice University, and it’s our mission to improve student access to education. Our first openly licensed college textbook was published in 2012, and our library has since scaled to over 35 books for college and AP® courses used by hundreds of thousands of students. OpenStax Tutor, our low-cost personalized learning tool, is being piloted in college courses throughout the country. Through our partnerships with philanthropic foundations and our alliance with other educational resource organizations, OpenStax is breaking down the most common barriers to learning and empowering students and instructors to succeed. About OpenStax Resources Customization Psychology 2eis licensed under a Creative Commons Attribut

In [34]:
max_data_count = 0
max_data_index = 0
more_than_256_count = 0
for val in tqdm(section_page_data):
    token_count = len(embed_model.tokenizer(val["context"], add_special_tokens=True, truncation=False)["input_ids"])
    if token_count > max_data_count:
        max_data_count = token_count
        max_data_index = section_page_data.index(val)
    if token_count > 256:
        more_than_256_count += 1


print(f"Max data count: {max_data_count}, Max data index: {max_data_index}")
print(f"Number of data points with more than 256 tokens: {more_than_256_count}")

100%|██████████| 764/764 [00:01<00:00, 442.38it/s]

Max data count: 1051, Max data index: 158
Number of data points with more than 256 tokens: 604


In [ ]:
chunk_sizes = [600, 700, 800, 900]

In [ ]:
import re
import random
from collections import defaultdict

random.seed(42)

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def first_sentence(text: str, max_chars: int = 220) -> str:
    text = clean_text(text)
    if not text:
        return ""
    parts = re.split(r"(?<=[.!?])\s+", text)
    s = parts[0] if parts else text
    return s[:max_chars].strip()

# group records by section so questions are balanced across topics
by_section = defaultdict(list)
for rec in section_page_data:
    section_name = rec.get("metadata", {}).get("sub_heading", "Unknown Section")
    by_section[section_name].append(rec)

question_bank = []
qid = 1

for section_name, rows in by_section.items():
    sample_n = min(2, len(rows))  # 2 questions per section (adjust if needed)
    picks = random.sample(rows, sample_n)

    for rec in picks:
        heading = rec.get("metadata", {}).get("heading", "")
        sub_heading = rec.get("metadata", {}).get("sub_heading", "")
        page = rec.get("metadata", {}).get("page", "")
        context = rec.get("context", "")
        hint = first_sentence(context)

        q1 = f"What is the main idea of {sub_heading} in {heading}?"
        q2 = f"Explain a key concept discussed in {sub_heading}."

        for q in (q1, q2):
            question_bank.append({
                "id": qid,
                "question": q,
                "gold_page": page,
                "gold_heading": heading,
                "gold_sub_heading": sub_heading,
                "hint": hint
            })
            qid += 1

# optional: shuffle and keep first N
random.shuffle(question_bank)
max_questions = 120
question_bank = question_bank[:max_questions]

with open("../Data/eval_queries_auto.json", "w", encoding="utf-8") as f:
    json.dump(question_bank, f, indent=2, ensure_ascii=False)

print(f"Generated {len(question_bank)} questions")
print("Saved to ../Data/eval_queries_auto.json")
question_bank[:3]

In [22]:
client = chromadb.Client()  

if "psychology-textbook" in client.list_collections():
    client.delete_collection(name="psychology-textbook")

database = client.get_or_create_collection(name="psychology-textbook")
for record in tqdm(section_page_data):
    database.add(
        ids=[str(record["id"])],
        metadatas=[record["metadata"]],
        documents=[record["context"]]
    )


  0%|          | 0/764 [00:00<?, ?it/s]

C:\Users\Peeyush\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:11<00:00, 7.49MiB/s]
100%|██████████| 764/764 [05:40<00:00,  2.24it/s]


In [ ]:
from transformers import AutoTokenizer
from tqdm import tqdm
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")  # or a similar model


max_token_length = 0
times_exceeded = 0
for record in tqdm(section_page_data):
    try: 
        tokens = tokenizer.encode(record["context"], return_tensors="pt", truncation=False)
    except Exception as e:
        times_exceeded += 1
print(f"Max token length: {max_token_length}. Times exceeded: {times_exceeded}")

  0%|          | 0/798 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (728 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (728 > 512). Running this sequence through the model will result in indexing errors
100%|██████████| 798/798 [00:05<00:00, 134.52it/s]

Max token length: 1091. Times exceeded 512 tokens: 395


In [26]:
text = open("../Data/Psychology2e_WEB_pdfminer_trimmed_with_page_numbers_v2.txt", "r", encoding="utf-8").read()
print(len(text))

1891667


In [28]:
print("max_seq_length:", embed_model.max_seq_length)

tok = embed_model.tokenizer
full_len = tok(text, add_special_tokens=True, truncation=False)["input_ids"]
trunc_len = tok(text, add_special_tokens=True, truncation=True, max_length=embed_model.max_seq_length)["input_ids"]

print("full tokens:", full_len)
print("used by model:", trunc_len)

max_seq_length: 256
full tokens: [101, 3931, 1015, 18443, 1015, 18443, 6160, 2000, 6825, 1016, 2063, 1010, 2019, 7480, 2696, 2595, 7692, 1012, 2023, 16432, 2001, 2517, 2000, 3623, 3076, 3229, 2000, 2152, 1011, 3737, 4083, 4475, 1010, 8498, 3284, 4781, 1997, 3834, 19838, 2953, 2012, 2210, 2000, 2053, 3465, 1012, 2055, 7480, 2696, 2595, 7480, 2696, 2595, 2003, 1037, 14495, 2241, 2012, 5785, 2118, 1010, 1998, 2009, 1521, 1055, 2256, 3260, 2000, 5335, 3076, 3229, 2000, 2495, 1012, 2256, 2034, 10132, 7000, 2267, 16432, 2001, 2405, 1999, 2262, 1010, 1998, 2256, 3075, 2038, 2144, 18953, 2000, 2058, 3486, 2808, 2005, 2267, 1998, 9706, 29656, 5352, 2109, 2011, 5606, 1997, 5190, 1997, 2493, 1012, 7480, 2696, 2595, 14924, 1010, 2256, 2659, 1011, 3465, 3167, 3550, 4083, 6994, 1010, 2003, 2108, 27220, 1999, 2267, 5352, 2802, 1996, 2406, 1012, 2083, 2256, 13797, 2007, 25321, 10100, 1998, 2256, 4707, 2007, 2060, 4547, 7692, 4411, 1010, 7480, 2696, 2595, 2003, 4911, 2091, 1996, 2087, 2691, 13500, 2000

In [33]:
len(full_len), len(trunc_len)

(381767, 256)

In [24]:
embed_model.encode(text, truncate=False)

array([ 2.49824058e-02,  1.02449534e-02, -2.57054041e-03,  4.51224968e-02,
        6.21668473e-02,  6.87291939e-03,  2.45948359e-02, -5.57248062e-03,
        3.57238837e-02,  4.04121131e-02,  1.35065690e-02,  5.77118136e-02,
        2.35685334e-02, -7.24258414e-03,  5.52421845e-02, -1.29821745e-03,
       -4.31844586e-04,  2.21410897e-02, -4.18858752e-02, -1.53773120e-02,
        3.41654308e-02,  2.58231964e-02,  8.32158774e-02, -1.10597201e-02,
       -9.37801506e-03,  4.31904159e-02, -2.26256941e-02, -6.22215085e-02,
        2.60887649e-02, -8.15225691e-02,  4.30139229e-02,  3.89125422e-02,
        3.09653245e-02, -1.49696348e-02,  2.63972636e-02, -7.17007043e-03,
       -1.42541090e-02,  1.10598896e-02,  3.18511017e-03,  8.62736441e-03,
       -6.50171265e-02, -2.73939762e-02, -4.03104685e-02,  7.08729550e-02,
       -6.85102791e-02, -7.79420286e-02,  1.20401829e-02, -1.14343047e-01,
        2.52268910e-02,  1.67670324e-02, -9.37166661e-02, -6.87824190e-02,
       -6.04896024e-02, -